# Parliamentary NLP — Hierarchy & Imbalance Experiments

Reproducible Colab/GPU notebook for detecting **offensive language and hate speech** in Brazilian Chamber of Deputies speeches.

**Uniform protocol** across all study models:

| Model | Checkpoint |
|--------|------------|
| TF-IDF+LR | classical lexical baseline |
| mBERT | `bert-base-multilingual-cased` |
| RoBERTa | `roberta-base` |
| BERTimbau | `neuralmind/bert-base-portuguese-cased` |

**Phases:** data hygiene → hierarchy / flat / cascade → Focal Loss + sampler → back-translation → speaker-level group split → report (timings, confusion matrices, per-class F1, ROC/PR, heatmaps).

Published artifacts from the full run are checked into the repo:

- [`docs/results/`](../docs/results/) — CSV + JSON metrics
- [`docs/figures/`](../docs/figures/) — heatmaps, bars, confusion matrices, ROC/PR
- Spec: [`docs/MODELING.md`](../docs/MODELING.md)

## How to run (Colab + GPU)

1. Runtime → Change runtime type → **GPU**
2. Upload this folder (`experimentos_pipeline.py` + this notebook) **or** clone the repo
3. Run cells in order (or only the `run_all` cell)
4. Download the generated `outputs/` directory


## 1) Instalação (Colab)


In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
print("IN_COLAB:", IN_COLAB)

if IN_COLAB:
    # GPU check
    !nvidia-smi -L || echo "Sem GPU — ative Runtime > Change runtime type > GPU"

    # Nao atualizar pandas para 3.x: conflita com google-colab e cudf (exige ~2.2.x)
    !pip -q install -U transformers sentencepiece openpyxl scikit-learn seaborn matplotlib
    !pip -q install "pandas==2.2.2"
    # torch ja vem no Colab; se faltar:
    # !pip -q install torch


## 2) Carregar o pipeline


In [ ]:
from pathlib import Path
import sys

# ---- Choose ONE option ----
# A) Files already in the current folder (Colab Files upload / local notebooks/)
# B) Google Drive
USE_DRIVE = False
DRIVE_PATH = "/content/drive/MyDrive/parliamentary-nlp-experiments"  # adjust if needed

if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    sys.path.insert(0, DRIVE_PATH)
    PIPELINE_DIR = Path(DRIVE_PATH)
else:
    PIPELINE_DIR = Path(".").resolve()

assert (PIPELINE_DIR / "experimentos_pipeline.py").exists(), (
    "Missing experimentos_pipeline.py. "
    "Keep it next to this notebook (repo: notebooks/), or mount Drive."
)

if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

import importlib
import experimentos_pipeline as exp
importlib.reload(exp)
print("Pipeline OK |", PIPELINE_DIR)


## 3) Configuração experimental


In [ ]:
# =========================
# CONFIGURE HERE
# =========================
exp.QUICK_MODE = False          # True = smoke test (2 folds, 1 model)
exp.RUN_TRANSFORMERS = True
exp.RUN_BACKTRANSLATION = True  # MarianMT; slower
exp.RUN_GROUP_SPLIT = True
exp.SAVE_FIGURES = True
exp.OUTPUT_DIR = Path("outputs")

exp.N_SPLITS = 5
exp.EPOCHS = 12
exp.BATCH_SIZE = 8              # raise to 16 if GPU memory allows
exp.PATIENCE = 3
exp.LR = 2e-5
exp.MAX_LENGTH = 128

# Full model suite (default). Restrict while debugging:
exp.MODELS = dict(exp.FULL_MODELS)
# exp.MODELS = {"BERTimbau": exp.FULL_MODELS["BERTimbau"]}

if exp.QUICK_MODE:
    exp.N_SPLITS = 2
    exp.EPOCHS = 3
    exp.RUN_BACKTRANSLATION = False
    exp.MODELS = {"BERTimbau": exp.FULL_MODELS["BERTimbau"]}

exp.set_seed(exp.SEED)
exp.print_config()


## 4) Execução

**Opção A — tudo de uma vez** (recomendado na noite / Colab Pro):


In [ ]:
# Roda fases 0-4 + tabela + figuras + tempos
results_df = exp.run_all(phases=[0, 1, 2, 3, 4])
results_df.head(20)


**Opção B — por fase** (se quiser interromper / retomar):


In [ ]:
# Descomente o que quiser rodar:

# exp.ALL_RESULTS.clear()
# exp.ALL_ARTIFACTS.clear()
# df_clean = exp.load_and_clean()
# exp.run_fase1(df_clean)   # hierarquia + flat + cascata (todos os modelos)
# exp.run_fase2(df_clean)   # focal + sampler
# exp.run_fase3(df_clean)   # back-translation
# exp.run_fase4(df_clean)   # group split por deputado
# results_df = exp.build_final_table()


## 5) Generated report

Besides confusion matrices, the pipeline writes under `outputs/`:

- CSV + JSON with Macro-F1, MCC, Precision/Recall, ROC-AUC/AP (binary), **runtime**
- Heatmaps experiment × model (F1, MCC, time)
- Macro-F1 and runtime bar charts
- Count + normalized confusion matrices
- Per-class Precision/Recall/F1
- ROC and Precision-Recall curves (binary tasks)
- `meta_execucao.json` with total wall time

After a successful full run, copy curated tables/figures into `docs/results/` and `docs/figures/` for the GitHub portfolio.


In [ ]:
from pathlib import Path
import pandas as pd

out = Path(exp.OUTPUT_DIR)
print("Arquivos gerados:")
for p in sorted(out.rglob("*")):
    if p.is_file():
        print(" -", p, f"({p.stat().st_size/1024:.1f} KB)")

csv = out / "resultados_experimentos_hierarquia_desbalanceamento.csv"
if csv.exists():
    df = pd.read_csv(csv)
    display_cols = [c for c in [
        "experimento", "modelo", "macro_f1_str", "mcc",
        "pos_f1", "roc_auc", "tempo_str", "tempo_minutos"
    ] if c in df.columns]
    display(df[display_cols].head(30))


In [ ]:
# Download on Colab
if IN_COLAB:
    import shutil
    from google.colab import files
    zip_path = shutil.make_archive("outputs_experiments", "zip", exp.OUTPUT_DIR)
    files.download(zip_path)
    print("Download started:", zip_path)
